# 🚗 Car Price Prediction – Jupyter Notebook

**An end-to-end ML analysis notebook for the Car Price Prediction project.**

---

### Sections
1. Setup & Imports
2. Data Loading
3. Data Cleaning
4. Exploratory Data Analysis
5. Feature Engineering
6. Model Training & Comparison
7. Best Model Evaluation
8. Prediction Example

## 1. Setup & Imports

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor': '#0b0b0d',
    'axes.facecolor':   '#1a1a2e',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2a2a4a',
    'axes.titlecolor':  '#ff7a00',
})

print('✅ All libraries imported successfully!')

## 2. Data Loading

In [ ]:
df = pd.read_csv('../dataset/car_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Data Types:')
df.info()
print('\nMissing Values:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')

## 3. Data Cleaning

In [ ]:
def clean_numeric(series):
    def _extract(val):
        if pd.isna(val): return np.nan
        m = re.search(r'[\d.]+', str(val))
        return float(m.group()) if m else np.nan
    return series.apply(_extract)

# Drop duplicates
df.drop_duplicates(inplace=True)

# Clean unit-bearing columns
for col in ['Mileage', 'Engine', 'Max_Power']:
    if col in df.columns:
        df[col] = clean_numeric(df[col])

# Fill missing values
for col in ['Mileage', 'Engine', 'Max_Power', 'Seats']:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)

for col in ['Fuel_Type', 'Transmission', 'Seller_Type', 'Owner']:
    if col in df.columns:
        df[col].fillna(df[col].mode()[0], inplace=True)

df.dropna(subset=['Selling_Price', 'Year', 'Kilometers_Driven'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Clean shape: {df.shape}')
df.describe()

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Car Dataset EDA', color='#ff7a00', fontsize=16, fontweight='bold')

# Price distribution
sns.histplot(df['Selling_Price']/1e5, bins=25, color='#ff7a00', kde=True, ax=axes[0,0])
axes[0,0].set_title('Price Distribution (Lakhs ₹)', color='#ff7a00')

# Fuel type
df['Fuel_Type'].value_counts().plot(kind='bar', color='#4fc3f7', ax=axes[0,1], edgecolor='none')
axes[0,1].set_title('Fuel Type Count', color='#ff7a00')
axes[0,1].tick_params(axis='x', rotation=30)

# Transmission
df['Transmission'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['#ff7a00','#4fc3f7','#81c784','#ce93d8'],
                                       ax=axes[1,0], ylabel='')
axes[1,0].set_title('Transmission Distribution', color='#ff7a00')

# Scatter: Year vs Price
axes[1,1].scatter(df['Year'], df['Selling_Price']/1e5, color='#ff7a00', alpha=0.5, s=30)
axes[1,1].set_title('Year vs Price (Lakhs ₹)', color='#ff7a00')
axes[1,1].set_xlabel('Year')
axes[1,1].set_ylabel('Selling Price (Lakhs)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
num_cols = df.select_dtypes(include=np.number).columns
fig, ax = plt.subplots(figsize=(10, 8), facecolor='#0b0b0d')
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='YlOrRd',
            mask=np.triu(np.ones_like(df[num_cols].corr(), dtype=bool)), ax=ax)
ax.set_title('Feature Correlation Heatmap', color='#ff7a00', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
df2 = df.copy()
df2['Car_Age'] = 2024 - df2['Year']
df2.drop(columns=['Model', 'Year'], errors='ignore', inplace=True)

cat_cols = [c for c in ['Brand','Fuel_Type','Transmission','Seller_Type','Owner'] if c in df2.columns]
num_cols = [c for c in ['Car_Age','Mileage','Engine','Max_Power','Seats','Kilometers_Driven'] if c in df2.columns]

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df2[col] = le.fit_transform(df2[col].astype(str))
    encoders[col] = le

feature_cols = num_cols + cat_cols
X = df2[feature_cols].copy()
y = df2['Selling_Price'].copy()

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
X.head()

## 6. Model Training & Comparison

In [ ]:
models = {
    'Linear Regression':          LinearRegression(),
    'Decision Tree':              DecisionTreeRegressor(random_state=42, max_depth=10),
    'Random Forest':              RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':          GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'MAE' : round(mean_absolute_error(y_test, preds), 0),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)), 0),
        'R2'  : round(r2_score(y_test, preds), 4),
    }
    print(f'{name:<32} R²={results[name]["R2"]:.4f}  MAE=₹{results[name]["MAE"]:,.0f}  RMSE=₹{results[name]["RMSE"]:,.0f}')

best = max(results, key=lambda k: results[k]['R2'])
print(f'\n★ Best Model: {best}')

In [ ]:
# Comparison chart
names = list(results.keys())
r2s   = [results[n]['R2'] for n in names]
colors = ['#4fc3f7','#81c784','#ff7a00','#ce93d8']

fig, ax = plt.subplots(figsize=(10,5), facecolor='#0b0b0d')
ax.set_facecolor('#1a1a2e')
bars = ax.bar(names, r2s, color=colors, edgecolor='none', width=0.5)
for b, v in zip(bars, r2s):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+.005, f'{v:.4f}',
            ha='center', va='bottom', color='white')
ax.set_ylim(0, 1.1)
ax.set_title('Model Comparison – R² Score', fontsize=15, color='#ff7a00', fontweight='bold')
ax.set_ylabel('R² Score', color='#e0e0e0')
ax.tick_params(axis='x', rotation=15, colors='#e0e0e0')
ax.tick_params(axis='y', colors='#e0e0e0')
ax.grid(axis='y', color='#2a2a4a', linewidth=0.5)
plt.tight_layout()
plt.show()

## 7. Best Model Evaluation

In [ ]:
best_model = models[best]
preds = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0b0b0d')
for ax in axes: ax.set_facecolor('#1a1a2e')

# Actual vs Predicted
axes[0].scatter(y_test/1e5, preds/1e5, color='#ff7a00', alpha=0.5, s=25)
mn, mx = min(y_test.min(), preds.min())/1e5, max(y_test.max(), preds.max())/1e5
axes[0].plot([mn, mx], [mn, mx], 'w--', linewidth=1.5)
axes[0].set_title(f'{best} – Actual vs Predicted', color='#ff7a00', fontweight='bold')
axes[0].set_xlabel('Actual Price (Lakhs)')
axes[0].set_ylabel('Predicted Price (Lakhs)')

# Residuals
residuals = (y_test.values - preds) / 1e5
axes[1].scatter(preds/1e5, residuals, color='#4fc3f7', alpha=0.5, s=25)
axes[1].axhline(0, color='white', linewidth=1.5, linestyle='--')
axes[1].set_title('Residual Plot', color='#ff7a00', fontweight='bold')
axes[1].set_xlabel('Predicted Price (Lakhs)')
axes[1].set_ylabel('Residual (Lakhs)')

plt.tight_layout()
plt.show()

print(f'MAE  : ₹ {mean_absolute_error(y_test, preds):,.0f}')
print(f'RMSE : ₹ {np.sqrt(mean_squared_error(y_test, preds)):,.0f}')
print(f'R²   : {r2_score(y_test, preds):.4f}')

## 8. Example Prediction

In [ ]:
# Example: Maruti Swift 2019, Petrol, Manual, 22 kmpl, 1197 CC, 82 bhp, 5 seats, 25000 km
sample = {
    'Car_Age'          : 2024 - 2019,  # 5
    'Mileage'          : 22.0,
    'Engine'           : 1197.0,
    'Max_Power'        : 82.0,
    'Seats'            : 5.0,
    'Kilometers_Driven': 25000.0,
    'Brand'            : encoders['Brand'].transform(['Maruti'])[0],
    'Fuel_Type'        : encoders['Fuel_Type'].transform(['Petrol'])[0],
    'Transmission'     : encoders['Transmission'].transform(['Manual'])[0],
    'Seller_Type'      : encoders['Seller_Type'].transform(['Dealer'])[0],
    'Owner'            : encoders['Owner'].transform(['First Owner'])[0],
}

import numpy as np
row = np.array([[sample[c] for c in feature_cols]])
row[:, :len(num_cols)] = scaler.transform(row[:, :len(num_cols)])

price = best_model.predict(row)[0]
print(f'🔮 Predicted Price: ₹ {price:,.0f}  ({price/1e5:.2f} Lakhs)')